# Performance: what is worth doing

Mag$\nu$s is fast enough for most single calls -- the median across 164 Earth and solar
configurations is 2 ms. Scans are what the code mostly does, and three things make them
substantially faster **without changing any answer**.

Every measurement here is taken live. Timings vary between machines and between runs, so treat
the ratios rather than the absolute numbers as the result, and note that two of the three
optimisations are worth nothing at all in the wrong circumstances -- which is more useful to
know than a headline speed-up.

In [1]:
# The figures are set through LaTeX where one is available; where it is not,
# matplotlib's own mathtext renders the labels instead.  Same numbers either way.
import shutil

import matplotlib.pyplot as plt

plt.rcParams['text.usetex'] = shutil.which('latex') is not None

In [2]:
import time
import warnings

import numpy as np

# np.trapz was removed in NumPy 2.0 and renamed np.trapezoid.  Ask the installed
# version which one it has rather than pinning either: these notebooks are read
# on whatever numpy the reader happens to have.
trapezoid = getattr(np, 'trapezoid', getattr(np, 'trapz', None))

# Mag(nu)s is imported as an installed package -- from the repository root,
# 'pip install -e .' (add [plot] for magnus.plotting). No sys.path juggling.
import magnus.magnus as magnus
import magnus.oscprob as oscprob
import magnus.hamiltonians as hamiltonians
import magnus.matter as matter
import magnus.earth as earth
import magnus.globaldefs as gd

warnings.simplefilter('ignore')

# load_nufit_params returns exactly the six mixing parameters, ready to splat
# into any osc_prob_3nu_* call.  'NuFIT 6.1' is the package default.
OSC = gd.load_nufit_params('NuFIT 6.1', 'NO')
osc = OSC
h_vac = np.asarray(hamiltonians.hamiltonian_3nu_vacuum_energy_independent(**OSC))
e00 = np.diag([1.0, 0.0, 0.0])

COSTHZ = -0.9
L_EARTH = earth.distance_traveled_inside_earth(COSTHZ)*gd.CONV_KM_TO_INV_EV

def best_of(call, repeats=3):
    """Fastest of a few runs -- the least noisy summary of a timing."""
    fastest, result = np.inf, None
    for _ in range(repeats):
        t0 = time.perf_counter()
        result = call()
        fastest = min(fastest, time.perf_counter() - t0)
    return result, fastest

## 1. Pass an array of energies, do not loop

Every wrapper accepts an array of energies, of baselines, or both. Handing the whole scan over
at once lets the matter profile be built once rather than once per point, which is what the
energy-batched engine exists to do.

In [3]:
def H(energy, l, VCC):
    return h_vac/energy + np.asarray(VCC)[..., None, None]*e00

energies = np.logspace(0.0, 1.5, 60)*gd.UNIT_GEV

batched, t_batched = best_of(lambda: np.asarray(oscprob.osc_prob_earth(
    H, energy=energies, costhz=COSTHZ, L=L_EARTH, nu_i=gd.NUMU, nu_f=gd.NUMU)))
looped, t_looped = best_of(lambda: np.array([
    oscprob.osc_prob_earth(H, energy=float(e), costhz=COSTHZ, L=L_EARTH,
                           nu_i=gd.NUMU, nu_f=gd.NUMU) for e in energies]))

print('%d energies through an Earth chord' % len(energies))
print('  array : %.3f s' % t_batched)
print('  loop  : %.3f s   (%.2fx slower)' % (t_looped, t_looped/t_batched))
print('  max |difference| = %.1e' % np.max(np.abs(batched - looped)))

60 energies through an Earth chord
  array : 0.041 s
  loop  : 0.120 s   (2.94x slower)
  max |difference| = 5.5e-06


The two answers are not bit-identical -- the batched path refines the profile once for
the whole scan rather than independently per point, so the two land at slightly different
grids. The difference is at the $10^{-6}$ level, far inside any tolerance you would request.

## 2. The palindrome, and when it is worth nothing

A chord through a spherically symmetric Earth meets every radius twice, so Mag$\nu$s can
evaluate your `H_func` on the first half of the slab chain and mirror the rest. That halves the
**number of positions** at which your Hamiltonian is evaluated.

Whether that is worth anything depends on a distinction worth being precise about: it halves
the positions, not the number of *calls*. If your `H_func` is dominated by fixed per-call
overhead, halving the positions saves nothing. If its cost scales with how many positions it
was handed -- an interpolation, a table lookup per point, a quadrature -- it saves half.

First, plain PREM, where a density lookup is too cheap to be worth halving:

In [4]:
def H_prem(energy, l, VCC):
    return h_vac/energy + np.asarray(VCC)[..., None, None]*e00

def timed(H_func, **kwargs):
    def call():
        return np.asarray(oscprob.osc_prob_earth(
            H_func, costhz=COSTHZ, L=L_EARTH, nu_i=gd.NUMU, nu_f=gd.NUMU, **kwargs))
    magnus.USE_PALINDROME = True
    on_result, on_time = best_of(call)
    magnus.USE_PALINDROME = False
    off_result, off_time = best_of(call)
    magnus.USE_PALINDROME = True                      # restore the default
    return on_time, off_time, np.max(np.abs(on_result - off_result))

on, off, gap = timed(H_prem, energy=5.0*gd.UNIT_GEV)
print('plain PREM lookup, single point')
print('  palindrome on  : %.4f s' % on)
print('  palindrome off : %.4f s   speed-up %.2fx' % (off, off/on))

plain PREM lookup, single point
  palindrome on  : 0.0019 s
  palindrome off : 0.0021 s   speed-up 1.10x


About 1.00x -- the bookkeeping costs as much as the lookups it saves. The package
documentation quotes 0.91x for this case, and disarming the optimisation here would cost you
nothing.

Now a Hamiltonian whose cost genuinely scales with the number of positions it is given. The
quadrature below stands in for an interpolated profile, a per-point integral, or any of the
things a real custom Hamiltonian does.

In [5]:
GRID = np.linspace(0.0, 1.0, 4000)

def H_expensive(energy, l, VCC):
    """Per-position work: a small quadrature for every position handed in."""
    V = np.atleast_1d(np.asarray(VCC, dtype=float))
    weight = trapezoid(np.exp(-GRID[None, :]*1.0e13*np.abs(V)[:, None]), GRID, axis=1)
    V_eff = V*(1.0 + 1.0e-12*weight)
    shape = np.asarray(VCC).shape
    V_eff = V_eff.reshape(shape) if shape else V_eff[0]
    return h_vac/energy + np.asarray(V_eff)[..., None, None]*e00

print('%-26s %-10s %-10s %-10s %s'
      % ('workload', 'on [s]', 'off [s]', 'speed-up', 'max |diff|'))
print('-'*66)
for label, kwargs in (('single point', dict(energy=5.0*gd.UNIT_GEV)),
                      ('20-energy scan',
                       dict(energy=np.logspace(0.0, 1.5, 20)*gd.UNIT_GEV))):
    on, off, gap = timed(H_expensive, **kwargs)
    print('%-26s %-10.4f %-10.4f %-10.2f %.1e' % (label, on, off, off/on, gap))

workload                   on [s]     off [s]    speed-up   max |diff|
------------------------------------------------------------------
single point               0.0092     0.0174     1.90       5.6e-16


20-energy scan             0.0872     0.2261     2.59       4.4e-16


Now it pays: the mirrored path is faster by a factor approaching two, and the answers
agree to round-off. The package documentation quotes 1.4--1.67x on an expensive `H_func`,
measured on a different profile; the numbers above are the same effect on this one.

The lesson is not "the palindrome is worth 1.75x". It is that **it is worth exactly half of
whatever your Hamiltonian charges per position, and nothing for what it charges per call**. If
you want it off, `magnus.magnus.USE_PALINDROME = False`.

Note also that only `osc_prob_earth` gets this: a chord is symmetric by geometry, and
`osc_prob_sun` deliberately does not declare it, because a solar profile is monotonic.

## 3. What a tolerance costs

Tightening `rtol` is not a smooth dial. The refinement ladder moves in steps, so several
requests can land on the same grid and cost exactly the same -- as notebook 21 showed, asking
for $10^{-4}$ sometimes buys the $10^{-3}$ answer for free, and sometimes the reverse.

In [6]:
ne = matter.exp_density_profile(gd.NUM_DENSITY_E_SUN_CENTRAL, gd.L_SCALE_SUN)
PARAMS_2NU = {'sth': 0.55, 'Dm2': 7.5e-5}
L_SUN = 0.5*gd.SUN_RADIUS*gd.UNIT_KM

print('%-14s %-10s %-12s %s' % ('rtol', 'time [s]', 'n_slabs', 'relative cost'))
print('-'*52)
baseline_time = None
for tol in (1.0e-2, 1.0e-3, 1.0e-4, 1.0e-5):
    info = {}
    _, t = best_of(lambda tol=tol, info=info: oscprob.osc_prob_matter_std_potential(
        2, ne, 1.0e7, L_SUN, PARAMS_2NU, L0=0.0,
        density_is_of_number_of_electrons=True,
        convergence_info=info, rtol=tol, atol=tol*1.0e-2), repeats=2)
    baseline_time = baseline_time or t
    print('%-14.0e %-10.3f %-12s %.2fx'
          % (tol, t, info['n_slabs'], t/baseline_time))

rtol           time [s]   n_slabs      relative cost
----------------------------------------------------
1e-02          0.011      6426         1.00x
1e-03          0.011      6426         1.00x
1e-04          0.017      9639         1.46x
1e-05          0.026      14458        2.27x


Three orders of magnitude of extra accuracy for roughly twice the work, and the first
of them free. Tolerances are usually worth tightening.

## Summary

| what | speed-up | when it is worth nothing |
|---|---|---|
| pass an array of energies | **~2.7x** | single-point calls |
| write `H_func` to take an array of positions | **~4.6x** (notebook 19) | never -- always do this |
| the palindrome, expensive `H_func` | **~1.8x** | cheap or per-call-dominated `H_func` |
| the palindrome, plain PREM | ~1.0x | this is the "worth nothing" case |
| tightening `rtol` by $10^{3}$ | costs ~2x | -- |

Ranked by what you control: **vectorise your `H_func` first** (notebook 19), **pass arrays
second**, and let the palindrome look after itself -- it is on by default, it is free when it
helps, and it costs a few percent when it does not.

None of these change an answer by more than round-off, which is the property that makes them
worth taking.

---

**Previous:** [When averaging rescues you](23_magnus_when_averaging_helps.ipynb)  
**Next:** [Against other codes](25_magnus_against_other_codes.ipynb) --- where a closed form wins, and a conventions trap that looks like accuracy  
[API reference](https://mbustama.github.io/Magnus/functions.html) &middot; [Implementation details](https://mbustama.github.io/Magnus/implementation_details.html) &middot; [All notebooks](.)